# **Dataset Preparation**

In [ ]:
import pandas as pd
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from imblearn.over_sampling import SMOTE

In [ ]:
import pandas as pd

df = pd.read_csv('/content/combined_data.csv')
df = df[["Sentences_clean", "unified_sentiment"]]
df.dropna(inplace=True)
df.columns = ["text", "label"]
label_map = {-1: 0, 0: 1, 1: 2}
df["label"] = df["label"].map(label_map)

print("Data loaded and labels mapped:")
print(df['label'].value_counts())
display(df.head())

Data loaded and labels mapped:
label
2    4940
0     715
1     405
Name: count, dtype: int64


,text,label
0,نصحوني بتجربه حمام الكبريت يمكنكم الدخول مع مج...,2
1,قلعه ساحره منظر خلاب للمدينه من اعلي القلعه يو...,2
2,تبليسي جورجيا من اجمل المدن التي زرتها في حيات...,2
3,جوله علي المدينه القديمه تبليسي شاردن ممتعه ال...,2
4,احلي اجازه لمحبي الطبيعه المناظر الخلابه الطبي...,2


# **Arabert**

In [ ]:
pip install transformers arabert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 12.1 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186456 sha256=78ff0dd48dbc0824f751e641aa9c4998fc51c406206c68052dad2808be2e5af1
  Stored in directory: /root/.cache/pip/wheels/bb/f1/26/f9002669ef6ad80a3c9f1b22880b35d9b4c6650011acee0523
Successfully built emoji


In [ ]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from arabert.preprocess import ArabertPreprocessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv('/content/combined_data.csv')
df = df[["Sentences_clean", "unified_sentiment"]].dropna()
df.columns = ["text", "label"]
label_map = {-1: 0, 0: 1, 1: 2}
df["label"] = df["label"].map(label_map)
model_id = 'aubmindlab/bert-base-arabertv02'
arabert_prep = ArabertPreprocessor(model_name=model_id)
df['text_processed'] = df['text'].apply(lambda x: arabert_prep.preprocess(x))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(df['text_processed'], df['label'], test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(model_id)
X_train_tokenized = tokenizer(list(X_train_raw), truncation=True, padding=True, return_tensors='pt')
X_test_tokenized = tokenizer(list(X_test_raw), truncation=True, padding=True, return_tensors='pt')

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(X_train_tokenized, y_train.values)
test_dataset = SentimentDataset(X_test_tokenized, y_test.values)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, predictions), 'f1': f1_score(labels, predictions, average='weighted')}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    report_to='none',
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print('Environment restored and Trainer ready for training.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Environment restored and Trainer ready for training.


In [ ]:
import torch
from sklearn.metrics import classification_report, accuracy_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def batch_predict(texts, batch_size=16):
    predictions = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, truncation=True, padding=True, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1).tolist()
            predictions.extend(preds)
    return predictions
print(f"Calculating accuracy on test set using {device}...")
test_texts = X_test_raw.tolist()
y_pred = batch_predict(test_texts)

acc = accuracy_score(y_test, y_pred)
print(f"\nOverall Accuracy: {acc:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

Calculating accuracy on test set using cuda...

Overall Accuracy: 0.8441

Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.39      0.09      0.14       124
     Neutral       0.00      0.00      0.00        62
    Positive       0.85      0.99      0.92      1026

    accuracy                           0.84      1212
   macro avg       0.42      0.36      0.35      1212
weighted avg       0.76      0.84      0.79      1212



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
